In [1]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("adult_with_headers.csv")

# Basic info
print(df.info())
print(df.describe())

# Check missing values
print(df.isnull().sum())

# Check unique values
for col in df.columns:
    print(col, df[col].nunique())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education_num   32561 non-null  int64 
 5   marital_status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital_gain    32561 non-null  int64 
 11  capital_loss    32561 non-null  int64 
 12  hours_per_week  32561 non-null  int64 
 13  native_country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB
None
                age        fnlwgt  education_num  capital_gain  capital_loss  \
count  3

In [2]:
# Replace '?' with NaN
df.replace('?', np.nan, inplace=True)

# Check again
print(df.isnull().sum())

age               0
workclass         0
fnlwgt            0
education         0
education_num     0
marital_status    0
occupation        0
relationship      0
race              0
sex               0
capital_gain      0
capital_loss      0
hours_per_week    0
native_country    0
income            0
dtype: int64


In [17]:
# Numerical columns
num_cols = df.select_dtypes(include=np.number).columns

# Categorical columns
cat_cols = df.select_dtypes(include='object').columns

# Fill numerical with median
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

# Fill categorical with mode
for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)

In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

In [7]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df[num_cols] = scaler.fit_transform(df[num_cols])

In [18]:
low_cardinality = [col for col in cat_cols if df[col].nunique() < 5]
high_cardinality = [col for col in cat_cols if df[col].nunique() >= 5]

In [9]:
df = pd.get_dummies(df, columns=low_cardinality, drop_first=True)

In [10]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

for col in high_cardinality:
    df[col] = le.fit_transform(df[col])

In [16]:
df['age_group'] = pd.cut(df['age'], bins=[0,25,50,100], labels=['Young','Adult','Senior'])

In [20]:
df['hours_per_age'] = df['hours_per_week'] / df['age']

In [19]:
print(df[num_cols].skew())

age                0.558743
workclass         -0.752024
fnlwgt             1.446980
education         -0.934042
education_num     -0.311676
marital_status    -0.013508
occupation         0.114583
relationship       0.786818
race              -2.435386
capital_gain      10.671437
capital_loss       4.594629
hours_per_week     0.227643
native_country    -3.658303
hours_per_age           NaN
dtype: float64


In [22]:
df['capital_gain'] = np.log1p(df['capital_gain'])